Some imports and paths

In [1]:
import os
import pandas as pd
from icecream import ic
from pathlib import Path
from pandas import DataFrame

THIS_FILE_PATH = Path(os.getcwd()) / "olistbr.ipynb"

THIS_PROJECT_PATH = THIS_FILE_PATH.parent.parent
DATA_RAW_PATH = THIS_PROJECT_PATH / "data" / "raw"

if not DATA_RAW_PATH.exists():
    raise FileExistsError()

ic(THIS_PROJECT_PATH)
ic(THIS_FILE_PATH)
ic(DATA_RAW_PATH)

ic| THIS_PROJECT_PATH: WindowsPath('c:/Users/alain/dev/personal/data-projects/dp-projects/static/olistbr')
ic| THIS_FILE_PATH: WindowsPath('c:/Users/alain/dev/personal/data-projects/dp-projects/static/olistbr/notebooks/olistbr.ipynb')
ic| DATA_RAW_PATH: WindowsPath('c:/Users/alain/dev/personal/data-projects/dp-projects/static/olistbr/data/raw')


WindowsPath('c:/Users/alain/dev/personal/data-projects/dp-projects/static/olistbr/data/raw')

In [2]:
raw_filenames: list[str] = os.listdir(DATA_RAW_PATH)
raw_filepaths: list[Path] = [DATA_RAW_PATH / f for f in raw_filenames]
# ic(raw_filenames)
# ic(raw_filepaths)

dfs: dict[str, DataFrame] = {f.name: pd.read_csv(f, dtype=str) for f in raw_filepaths}
# ic(dfs)

Measure Functions

In [3]:
def lev_d(a: str, b: str) -> int:
    """Levenshtein distance
    measures the distance between two strings from doing operations:
        insert
        remove
        replace

    time: O(mn) quadratic
    """
    lena = len(a)
    lenb = len(b)

    if lenb == 0:
        return lena

    if lena == 0:
        return lenb

    heada = a[0]
    headb = b[0]
    taila = a[1:]
    tailb = b[1:]

    if heada == headb:
        return lev_d(taila, tailb)

    return 1 + min(lev_d(taila, b), lev_d(a, tailb), lev_d(taila, tailb))


def sim_norm_lev_d(a: str, b: str) -> float:
    """similarity normalised lev_d"""
    lena = len(a)
    lenb = len(b)

    return 1 - lev_d(a, b) / max(lena, lenb)


def dice_sim(A: set[str], B: set[str]) -> float:
    """Dice similarity
    |AnB| / (|A|+|B|/2) = 2*|AnB| / (|A|+|B|) in [0,1]
    how much a set overlaps another set given their average size
    complete coverage = 1
    partial coverage = (0, 1)
    no coverage = 0
    """
    return 2 * len(A.intersection(B)) / (len(A) + len(B))

Inference Function

In [4]:
import string


def infer_dtypes(df: DataFrame) -> DataFrame:
    """
    df -> out_df
    originally intended to fully infer (end-to-end) a cols datatype from its attributes
    incomplete
    now used as an aid for the engineer to determine col datatypes from its attributes
    """

    def is_chars_in_string(chars: str, parent_string: str) -> bool:
        if set(chars).intersection(set(parent_string)):
            return True

        return False

    # TODO: add: chars_used_subset_of_hex_digits
    # TODO: add: is_numeric_float
    out_df = pd.DataFrame(
        index=df.columns,
        columns=[
            # keys
            "has_unique_entries",
            # nulls
            "has_nulls",
            "where_nulls",
            "total_nulls",
            # chars
            "sorted_chars_used",
            "total_unique_chars_used",
            "has_ascii",
            "has_non_ascii",
            "has_prefix_zero",
            "dice_sim_to_ascii",
            "dice_sim_to_non_ascii",
            "min_str_value",
            "max_str_value",
            "entry_lengths",
            "total_unique_entry_lengths",
            "max_entry_length",
            "is_fixed_length",
            # numeric
            "chars_used_subset_of_numeric",
            "has_prefix_dash",
            "has_digits",
            "has_hex_digits",
            "has_decimal",
            "dice_sim_to_digits",
            "dice_sim_to_hex_digits",
            "min_numeric_value",
            "max_numeric_value",
            # datetime
            "has_dash",
            "has_colon",
            "has_space",
            # bit
            "has_exactly_two_entries",
        ],
    )

    cols = df.columns

    for col in cols:
        print(col)

        # global vars
        clean_series = df[col].dropna()
        chars_used: set[str] = set("".join(clean_series.astype(str)))
        sorted_chars_used: str = "".join(sorted(chars_used))

        # keys ==================================================
        out_df.loc[col, "has_unique_entries"] = 1 if clean_series.is_unique else 0

        # nulls ==================================================
        where_null = df[col].isnull()

        out_df.loc[col, "has_nulls"] = 1 if where_null.any() else 0
        out_df.loc[col, "where_nulls"] = df[where_null].index.tolist()
        out_df.loc[col, "total_nulls"] = where_null.sum()

        # chars ==================================================
        # if col chars > ascii when col chars - ascii > 0
        excess_ascii: set[str] = chars_used - set(string.printable)
        entry_lengths = sorted(clean_series.astype(str).str.len().unique().tolist())

        out_df.loc[col, "sorted_chars_used"] = sorted_chars_used
        out_df.loc[col, "total_unique_chars_used"] = len(sorted_chars_used)
        out_df.loc[col, "has_ascii"] = (
            1
            if is_chars_in_string(
                string.printable,
                sorted_chars_used,
            )
            else 0
        )
        out_df.loc[col, "has_non_ascii"] = 1 if excess_ascii else 0
        out_df.loc[col, "has_prefix_zero"] = (
            1 if clean_series.astype(str).str.startswith("0").any() else 0
        )
        out_df.loc[col, "dice_sim_to_ascii"] = dice_sim(
            set(string.printable),
            chars_used,
        )
        out_df.loc[col, "dice_sim_to_non_ascii"] = dice_sim(
            excess_ascii,
            chars_used,
        )
        out_df.loc[col, "min_str_value"] = min(clean_series)
        out_df.loc[col, "max_str_value"] = max(clean_series)
        out_df.loc[col, "entry_lengths"] = entry_lengths
        out_df.loc[col, "total_unique_entry_lengths"] = len(entry_lengths)
        out_df.loc[col, "max_entry_length"] = max(entry_lengths)
        out_df.loc[col, "is_fixed_length"] = 1 if len(entry_lengths) == 1 else 0

        # numeric ==================================================
        str_numeric = string.digits + "-."

        out_df.loc[col, "chars_used_subset_of_numeric"] = (
            1 if chars_used.issubset(str_numeric) else 0
        )
        out_df.loc[col, "has_prefix_dash"] = (
            1 if clean_series.astype(str).str.startswith("-").any() else 0
        )
        out_df.loc[col, "has_digits"] = (
            1
            if is_chars_in_string(
                string.digits,
                sorted_chars_used,
            )
            else 0
        )
        out_df.loc[col, "has_hex_digits"] = (
            1
            if is_chars_in_string(
                string.hexdigits,
                sorted_chars_used,
            )
            else 0
        )
        out_df.loc[col, "has_decimal"] = (
            1
            if is_chars_in_string(
                ".",
                sorted_chars_used,
            )
            else 0
        )
        out_df.loc[col, "dice_sim_to_digits"] = dice_sim(
            set(string.digits),
            chars_used,
        )
        out_df.loc[col, "dice_sim_to_hex_digits"] = dice_sim(
            set(string.hexdigits),
            chars_used,
        )

        val = pd.to_numeric(clean_series, errors="coerce")
        numeric_min = val.min()
        numeric_max = val.max()
        out_df.loc[col, "min_numeric_value"] = (
            numeric_min if pd.notnull(numeric_min) else pd.NA
        )
        out_df.loc[col, "max_numeric_value"] = (
            numeric_max if pd.notnull(numeric_max) else pd.NA
        )

        # datetime ==================================================
        out_df.loc[col, "has_dash"] = (
            1
            if is_chars_in_string(
                "-",
                sorted_chars_used,
            )
            else 0
        )
        out_df.loc[col, "has_colon"] = (
            1
            if is_chars_in_string(
                ":",
                sorted_chars_used,
            )
            else 0
        )
        out_df.loc[col, "has_space"] = (
            1
            if is_chars_in_string(
                " ",
                sorted_chars_used,
            )
            else 0
        )

        # bit ==================================================
        unique_entries = clean_series.unique()
        out_df.loc[col, "has_exactly_two_entries"] = (
            1 if len(unique_entries) == 2 else 0
        )
    return out_df

Column analysis

In [5]:
pd.set_option("display.max_columns", None)

# filename = raw_filenames[6]
filename = "olist_" + "customers" + "_dataset.csv"
# filename = 'product_category_name_translation.csv'
print(filename)

df = dfs[filename]
df_attributes = infer_dtypes(df)
df_attributes
# print(df_attributes)

olist_customers_dataset.csv
customer_id
customer_unique_id
customer_zip_code_prefix
customer_city
customer_state


,has_unique_entries,has_nulls,where_nulls,total_nulls,sorted_chars_used,total_unique_chars_used,has_ascii,has_non_ascii,has_prefix_zero,dice_sim_to_ascii,dice_sim_to_non_ascii,min_str_value,max_str_value,entry_lengths,total_unique_entry_lengths,max_entry_length,is_fixed_length,chars_used_subset_of_numeric,has_prefix_dash,has_digits,has_hex_digits,has_decimal,dice_sim_to_digits,dice_sim_to_hex_digits,min_numeric_value,max_numeric_value,has_dash,has_colon,has_space,has_exactly_two_entries
customer_id,1,0,[],0,0123456789abcdef,16,1,0,1,0.275862,0.0,00012a2ce6f8dcda20d059ce98491703,ffffe8b65bbe3087b653a978c870db99,[32],1,32,1,0,0,1,1,0,0.769231,0.842105,<NA>,<NA>,0,0,0,0
customer_unique_id,0,0,[],0,0123456789abcdef,16,1,0,1,0.275862,0.0,0000366f3b9a7992bf8c76cfdf3221e2,ffffd2657e2aad2907e67c3e9daecbeb,[32],1,32,1,0,0,1,1,0,0.769231,0.842105,<NA>,<NA>,0,0,0,0
customer_zip_code_prefix,0,0,[],0,0123456789,10,1,0,1,0.181818,0.0,01003,99990,[5],1,5,1,1,0,1,1,0,1.0,0.625,1003,99990,0,0,0,0
customer_city,0,0,[],0,'-14abcdefghijklmnopqrstuvwxyz,31,1,0,0,0.473282,0.0,abadia dos dourados,zortea,"[3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, ...",28,32,0,0,0,1,1,0,0.097561,0.301887,<NA>,<NA>,1,0,1,0
customer_state,0,0,[],0,ABCDEFGIJLMNOPRST,17,1,0,0,0.290598,0.0,AC,TO,[2],1,2,1,0,0,0,1,0,0.0,0.307692,<NA>,<NA>,0,0,0,0


Scratch work

In [6]:
"""scratch
inclusive
tinyint:     2**0-1 to 2**8-1
smallint:   -2**15  to 2**15-1
int:        -2**31  to 2**31-1
bigint:     -2**63  to 2**63-1

money: -922,337,203,685,477.5808 to 922,337,203,685,477.5807 (-922,337,203,685,477.58
to 922,337,203,685,477.58 for Informatica. Informatica only supports two decimals, not four.)

smallmoney: -214,748.3648 to 214,748.3647
"""

"""
different known datatypes

numeric if chars_used_subset_of_numeric else other
    exact numerics
        (tiny, small, int, big)
        money
        smallmoney
    approximate numerics
        float
        real

datetime if has_dash and has_colon and has_space else other
    TODO: has_datetime_format
    datetime2

bit if has_exactly_two_entries else other
    TODO: has_bit_format: True/False, 1/0, true/false, yes/no
    bit

text is other 

text
numeric
datetime
bit

assume is text
(n, var, char) (#)
    'n' if has_non_ascii else ''
    '' if is_fixed_length else 'var'
    'char'
    # = max_entry_length

assume is int
(tiny, small, int, big)
    has_prefix_dash
    M = max(abs(min_numeric_value), abs(max_numeric_value))
    
    if min_numeric_value is negative: cannot be tinyint

    if M <= 2**8-1:

decimal(p,s)
datetime2
bit

not null
primary key

"""
print(
    df_attributes[
        [
            "chars_used_subset_of_numeric",
            "has_prefix_dash",
            "has_digits",
            "has_hex_digits",
            "has_decimal",
            "dice_sim_to_digits",
            "dice_sim_to_hex_digits",
            "min_numeric_value",
            "max_numeric_value",
        ]
    ]
)

                         chars_used_subset_of_numeric has_prefix_dash  \
customer_id                                         0               0   
customer_unique_id                                  0               0   
customer_zip_code_prefix                            1               0   
customer_city                                       0               0   
customer_state                                      0               0   

                         has_digits has_hex_digits has_decimal  \
customer_id                       1              1           0   
customer_unique_id                1              1           0   
customer_zip_code_prefix          1              1           0   
customer_city                     1              1           0   
customer_state                    0              1           0   

                         dice_sim_to_digits dice_sim_to_hex_digits  \
customer_id                        0.769231               0.842105   
customer_unique_id      